In [1]:
import anndata as ad
import pandas as pd

a = ad.read_h5ad("/workspace/combat.h5ad", backed="r")
print("shape:", a.shape)
print("\nobs columns:")
print(list(a.obs.columns))
print("\nvar columns:", list(a.var.columns))
print("\nlayers:", list(a.layers.keys()) if hasattr(a, 'layers') else None)
print("\nobsm:", list(a.obsm.keys()))

shape: (836148, 36306)

obs columns:
['GEX_region', 'cluster', 'cluster_source', 'minor_subset', 'minor_subset_source', 'major_subset', 'major_subset_source', 'cell_type_source', 'scRNASeq_sample_ID', 'QC_ngenes', 'QC_total_UMI', 'QC_pct_mitochondrial', 'QC_scrub_doublet_scores', 'TCR_clone_count', 'TCR_clone_proportion', 'BCR_total_mut_HC', 'BCR_clonal_abundance_HC', 'BCR_total_mut_LC', 'assay_ontology_term_id', 'disease_ontology_term_id', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'cell_type_original', 'donor_id', 'Source', 'Age', 'BMI', 'Hospitalstay', 'Death28', 'Institute', 'PreExistingHeartDisease', 'PreExistingLungDisease', 'PreExistingKidneyDisease', 'PreExistingDiabetes', 'PreExistingHypertension', 'PreExistingImmunocompromised', 'Smoking', 'Symptomatic', 'Requiredvasoactive', 'SARSCoV2PCR', 'Outcome', 'TimeSinceOnset', 'sex_ontology_term_id', 'development_stage_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'suspension_type', 'tissue_type', 'is_p

In [2]:
obs = a.obs[["donor_id","Source","disease","Age","BMI","Hospitalstay","Death28","Outcome",
             "Institute","SARSCoV2PCR","Symptomatic","TimeSinceOnset","Smoking",
             "Requiredvasoactive","sex","development_stage","scRNASeq_sample_ID",
             "GEX_region","major_subset","minor_subset","cell_type"]].copy()

donor_meta = obs.drop_duplicates("donor_id").drop(
    columns=["major_subset","minor_subset","cell_type"])
print("n donors:", len(donor_meta))
for col in ["Source","disease","Outcome","Death28","Institute","SARSCoV2PCR","Symptomatic","Smoking","sex"]:
    print(f"\n== {col} ==")
    print(donor_meta[col].value_counts(dropna=False).to_string())
print("\n== continuous (describe) ==")
print(donor_meta[["Age","BMI","Hospitalstay","TimeSinceOnset"]].describe().to_string())

n donors: 124

== Source ==
Source
COVID_SEV         30
Sepsis            23
COVID_CRIT        17
COVID_MILD        17
COVID_HCW_MILD    13
Flu               12
HV                10
COVID_LDN          2

== disease ==
disease
COVID-19     102
influenza     12
normal        10

== Outcome ==
Outcome
Intubated, ventilated       32
Not hospitalized            23
NaN                         23
Hospitalized, O2            13
Hospitalized, no O2         13
Death                       10
Non-invasive ventilation    10

== Death28 ==
Death28
False    108
True      16

== Institute ==
Institute
Oxford        110
St_Georges     14

== SARSCoV2PCR ==
SARSCoV2PCR
Positive    79
Not Done    45

== Symptomatic ==
Symptomatic
True    124

== Smoking ==
Smoking
never or unknown    90
prior               26
current              8

== sex ==
sex
male      73
female    51

== continuous (describe) ==
       Hospitalstay  TimeSinceOnset
count     87.000000      114.000000
mean       2.942529       10.2894

In [3]:
print("== major_subset ==")
print(obs["major_subset"].value_counts().to_string())
print("\n== minor_subset (n=%d) ==" % obs["minor_subset"].nunique())
print(obs["minor_subset"].value_counts().to_string())

# cells per donor x major_subset: how many donors have >= 50 cells per group?
ct = pd.crosstab(obs["donor_id"], obs["major_subset"])
print("\ndonors with >=50 cells per major_subset:")
print((ct >= 50).sum().to_string())

== major_subset ==
major_subset
CD4       269661
cMono     213697
CD8       106025
NK         69927
nan        52444
B          44256
ncMono     33172
GDT         9580
DC          8959
PB          8596
DP          6561
DN          4557
MAIT        4541
HSC         1868
PLT         1563
iNKT         372
RET          304
Mast          65

== minor_subset (n=41) ==
minor_subset
cMono                       207199
CD4.NAIVE                   123403
CD4.TCM                     107724
nan                          48341
NK.CD16hi                    48305
CD8.TEMRA                    33907
ncMono                       33172
CD8.NAIVE                    25451
B.NAIVE                      21279
CD8.TEM                      16813
CD4.TREG                     14195
B.INT                        11249
B.MEM                        10515
CD4.TEFF                     10303
CD4.TEM/TEMRA                 9261
PB                            8596
CD8.mitohi                    7957
CD8.TCM                    

In [4]:
import pandas as pd, numpy as np, os

BASE = "/mnt/shared-workspace/shared/combat"
MAT = f"{BASE}/mat"
os.makedirs(MAT, exist_ok=True)

counts = pd.read_parquet(f"{BASE}/pb_counts.parquet")   # genes x samples
coldata = pd.read_csv(f"{BASE}/pb_coldata.csv")
print("counts:", counts.shape, "| coldata:", coldata.shape)

N_HVG = 3000

def logcpm_df(c):  # c: genes x samples raw counts -> samples x genes log2 CPM
    lib = c.sum(axis=0)
    return (np.log2(c.div(lib, axis=1) * 1e6 + 1)).T

def prep_matrix(c, n_hvg=N_HVG, min_frac=0.10):
    keep = (c >= 1).mean(axis=1) >= min_frac
    c = c.loc[keep]
    lc = logcpm_df(c)
    v = lc.var(axis=0).sort_values(ascending=False)
    genes = v.index[:min(n_hvg, len(v))]
    return lc[genes]

cts = coldata["cell_type"].unique()
for ct in cts:
    cols = coldata.loc[coldata.cell_type == ct, "sample"]
    m = prep_matrix(counts[cols])
    m.index = [s.split("||")[0] for s in cols]  # donor_id rows
    m.to_csv(f"{MAT}/ct_{ct}.csv")
    print(f"{ct}: {m.shape[0]} donors x {m.shape[1]} genes")

# global pseudobulk (sum all CTs per donor)
donor = coldata["donor_id"].to_numpy()
glob = counts.T.groupby(donor).sum().T          # genes x donors
gm = prep_matrix(glob)
gm.to_csv(f"{MAT}/global.csv")
print("global:", gm.shape)

# grouped pseudobulk
GROUPS = {"T": ["CD4","CD8","GDT","DP"], "B_PB": ["B","PB"], "NK": ["NK"],
          "Mono": ["cMono","ncMono"], "DC": ["DC"]}
ct_of = coldata.set_index("sample")["cell_type"]
for gn, members in GROUPS.items():
    cols = [s for s in counts.columns if ct_of.get(s) in members]
    g = counts[cols].T.groupby([s.split("||")[0] for s in cols]).sum().T
    m = prep_matrix(g)
    m.to_csv(f"{MAT}/group_{gn}.csv")
    print(f"group {gn}: {m.shape}")

counts: (36306, 918) | coldata: (918, 26)


B: 109 donors x 3000 genes


CD4: 121 donors x 3000 genes


CD8: 119 donors x 3000 genes


DC: 67 donors x 3000 genes


...

group T: (121, 3000)


group B_PB: (111, 3000)


group NK: (118, 3000)


group Mono: (120, 3000)


group DC: (67, 3000)


In [5]:
import pandas as pd, numpy as np
from scipy.cluster import hierarchy
from scipy.spatial.distance import pdist

counts = pd.read_parquet("/mnt/shared-workspace/shared/combat/pb_counts.parquet")
var = pd.read_csv("/mnt/shared-workspace/shared/combat/pb_var.csv")
id2sym = dict(zip(var.feature_id, var.feature_name))

ct_of = pd.Series([c.split("||")[1] for c in counts.columns], index=counts.columns)
lib = counts.sum(axis=0)
lcpm = np.log2(counts.div(lib, axis=1) * 1e6 + 1)
prof = lcpm.T.groupby(ct_of).mean().T   # genes x CT

MARK = {"CD3D":"CD3D","CD4":"CD4","CD8A":"CD8A","CD14":"CD14","LST1":"LST1",
        "MS4A1":"MS4A1","GNLY":"GNLY","NCAM1":"NCAM1","SDC1":"SDC1","FCER1A":"FCER1A",
        "PPBP":"PPBP","HBA1":"HBA1"}
sym2id = {}
for gid, sym in id2sym.items():
    if sym in MARK.values() and sym not in sym2id:
        sym2id[sym] = gid
rows = [sym2id[s] for s in MARK.values() if s in sym2id]
mk = prof.loc[rows]
mk.index = [id2sym[g] for g in rows]
print("mean logCPM per CT (canonical markers):")
print(mk.round(2).to_string())

D = pdist(prof.T.values, metric="correlation")
Z = hierarchy.linkage(D, method="average")
order = hierarchy.leaves_list(Z)
print("\nlineage clustering order:", list(prof.columns[order]))

mean logCPM per CT (canonical markers):
            B   CD4   CD8    DC    DP    GDT     NK    PB  cMono  ncMono
CD3D     4.41  9.78  9.87  2.98  9.76   9.93   5.65  2.22   4.27    3.59
CD4      1.55  7.84  1.76  7.85  6.91   1.77   1.82  0.68   6.93    7.70
CD8A     3.35  3.95  9.92  1.53  9.33   7.05   7.39  1.53   2.97    2.37
CD14     3.75  4.11  3.96  7.19  3.10   3.52   3.90  2.21   9.84    8.29
LST1     6.28  6.03  5.60  9.13  5.81   5.65   5.02  2.30   9.99   10.47
MS4A1   10.39  3.34  4.32  2.49  3.43   3.97   2.83  3.30   2.73    2.16
GNLY     6.58  7.28  9.27  5.54  8.67  10.41  10.94  4.67   6.35    5.96
NCAM1    0.33  1.14  2.79  0.76  1.29   4.53   7.13  0.08   1.41    0.22
SDC1     0.16  0.25  0.15  0.00  0.08   0.08   0.13  5.54   0.16    0.05
FCER1A   0.52  0.85  0.59  8.67  0.15   0.38   0.56  0.02   1.87    0.30
PPBP     1.78  1.96  1.91  1.03  0.64   1.06   1.71  0.60   3.99    3.44
HBA1     0.42  0.58  0.89  0.17  0.49   1.05   2.43  0.12   0.77    0.49

lineage cl

In [6]:
import anndata as ad

for name, path in [("stephenson", "/workspace/stephenson.h5ad"),
                   ("hlca_core", "/workspace/hlca_core.h5ad"),
                   ("smillie", "/workspace/smillie.h5ad")]:
    b = ad.read_h5ad(path, backed="r")
    print(f"===== {name}: {b.shape} =====")
    print("obs:", list(b.obs.columns))
    b.file.close()
    print()

===== stephenson: (647366, 24245) =====
obs: ['sample_id', 'n_genes', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'initial_clustering', 'Resample', 'Collection_Day', 'Swab_result', 'Status', 'Smoker', 'Status_on_day_collection', 'Status_on_day_collection_summary', 'Days_from_onset', 'Site', 'time_after_LPS', 'Worst_Clinical_Status', 'Outcome', 'donor_id', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'author_cell_type', 'suspension_type', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid']



===== hlca_core: (584944, 27402) =====
obs: ['suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_ann_level_1', 'original_ann_level_2', 'original_ann_level_3'

In [7]:
import anndata as ad, pandas as pd

def label_report(path, name, label_cols, donor_col="donor_id"):
    b = ad.read_h5ad(path, backed="r")
    o = b.obs[[donor_col] + label_cols].copy()
    b.file.close()
    print(f"===== {name}: {o[donor_col].nunique()} donors =====")
    for lc in label_cols:
        ct = pd.crosstab(o[donor_col], o[lc].astype(object))
        well = (ct >= 50).sum()
        print(f"-- {lc}: {ct.shape[1]} labels; labels with >=50 donors at >=50 cells: "
              f"{sorted(well[well >= 30].index.tolist())}")
    return o

o_st = label_report("/workspace/stephenson.h5ad", "stephenson", ["cell_type", "author_cell_type"])
print()
o_sm = label_report("/workspace/smillie.h5ad", "smillie", ["cell_type", "CellType"])

===== stephenson: 120 donors =====
-- cell_type: 46 labels; labels with >=50 donors at >=50 cells: ['CD14-positive monocyte', 'CD16-negative, CD56-bright natural killer cell, human', 'CD16-positive, CD56-dim natural killer cell, human', 'T follicular helper cell', 'T-helper 22 cell', 'central memory CD4-positive, alpha-beta T cell', 'class switched memory B cell', 'effector CD8-positive, alpha-beta T cell', 'effector memory CD8-positive, alpha-beta T cell', 'gamma-delta T cell', 'immature B cell', 'mature NK T cell', 'mucosal invariant T cell', 'naive B cell', 'naive thymus-derived CD4-positive, alpha-beta T cell', 'naive thymus-derived CD8-positive, alpha-beta T cell', 'natural killer cell', 'platelet']


-- author_cell_type: 51 labels; labels with >=50 donors at >=50 cells: ['B_immature', 'B_naive', 'B_switched_memory', 'CD14_mono', 'CD16_mono', 'CD4.CM', 'CD4.IL22', 'CD4.Naive', 'CD4.Tfh', 'CD8.EM', 'CD8.Naive', 'CD8.TE', 'CD83_CD14_mono', 'MAIT', 'NK_16hi', 'NK_56hi', 'NK_prolif', 'Platelets', 'gdT']

===== smillie: 12 donors =====
-- cell_type: 23 labels; labels with >=50 donors at >=50 cells: []
-- CellType: 38 labels; labels with >=50 donors at >=50 cells: []


In [8]:
b = ad.read_h5ad("/workspace/smillie.h5ad", backed="r")
o = b.obs[["donor_id","disease","Source","tissue","suspension_type"]].copy()
b.file.close()
print(o.drop_duplicates("donor_id")[["donor_id","disease","Source","tissue"]].to_string(index=False))
print("\ncells per donor:")
print(o.groupby(["donor_id","disease"]).size().to_string())

donor_id disease  Source                     tissue
     N10  normal N10;Epi         colonic epithelium
      N8  normal  N8;Epi         colonic epithelium
     N11  normal N11;Epi         colonic epithelium
     N13  normal N13;Epi         colonic epithelium
     N15  normal N15;Epi         colonic epithelium
     N16  normal N16;Epi         colonic epithelium
     N17  normal N17;Epi mucosa of transverse colon
     N18  normal N18;Epi mucosa of transverse colon
     N20  normal N20;Epi         colonic epithelium
     N21  normal N21;Epi         colonic epithelium
     N51  normal N51;Epi          caecum epithelium
     N46  normal N46;Epi         colonic epithelium

cells per donor:
donor_id  disease
N8        normal      694
N10       normal     5919
N11       normal     4268
N13       normal      761
N15       normal     4720
N16       normal     2187
N17       normal     1664
N18       normal     2985
N20       normal     2182
N21       normal     3404
N46       normal     1692
N5

/tmp/ipykernel_124/3853604082.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(o.groupby(["donor_id","disease"]).size().to_string())


In [9]:
b = ad.read_h5ad("/workspace/hlca_core.h5ad", backed="r")
o = b.obs[["donor_id","cell_type","disease","smoking_status","study","sequencing_platform",
           "tissue_level_2","subject_type","scanvi_label","reannotation_type"]].copy()
b.file.close()
print("donors:", o.donor_id.nunique())
print("\n== disease (donors) ==")
print(o.drop_duplicates("donor_id")["disease"].astype(str).value_counts().to_string())
print("\n== smoking_status (donors) ==")
print(o.drop_duplicates("donor_id")["smoking_status"].astype(str).value_counts().to_string())
print("\nn cell_type labels:", o.cell_type.nunique(), "| n scanvi:", o.scanvi_label.nunique(),
      "| n reannotation:", o.reannotation_type.nunique())
ct = pd.crosstab(o.donor_id, o.cell_type.astype(str))
well = (ct >= 50).sum()
print("\ncell_type labels with >=40 donors at >=50 cells:")
print(sorted(well[well >= 40].index.tolist()))

donors: 107

== disease (donors) ==
disease
normal    107

== smoking_status (donors) ==
smoking_status
never     51
active    28
former    19
nan        9

n cell_type labels: 50 | n scanvi: 29 | n reannotation: 3



cell_type labels with >=40 donors at >=50 cells:
['CD1c-positive myeloid dendritic cell', 'CD8-positive, alpha-beta T cell', 'alveolar macrophage', 'classical monocyte', 'elicited macrophage', 'multiciliated columnar cell of tracheobronchial tree', 'pulmonary alveolar type 2 cell', 'respiratory basal cell', 'vein endothelial cell']


In [10]:
sites = o.groupby("donor_id")["tissue_level_2"].nunique()
print("donors by #distinct tissue_level_2 sites:")
print(sites.value_counts().to_string())
print("\nsite distribution (cells):")
print(o["tissue_level_2"].astype(str).value_counts().to_string())
print("\nstudy (donors):")
print(o.drop_duplicates("donor_id")["study"].astype(str).value_counts().to_string())
print("\nsequencing_platform (donors):")
print(o.drop_duplicates("donor_id")["sequencing_platform"].astype(str).value_counts().to_string())

donors by #distinct tissue_level_2 sites:
tissue_level_2
1    54
0    35
2    10
4     5
3     3

site distribution (cells):
tissue_level_2
nan                             134558
parenchyma lower lobe           118634
inferior turbinate               78279
lobular bronchi                  74464
parenchyma upper lobe            61735
trachea                          54004
distal lobular airways           27543
parenchyma right middle lobe     18541
left or right main bronchus       9417
segmental bronchi                 7769

study (donors):
study
Banovich_Kropski_2020     38
Seibold_2020              14
Nawijn_2021               10
Barbry_Leroy_2020         10
Misharin_Budinger_2018     8
Jain_Misharin_2021         7
Lafyatis_Rojas_2019        6
Teichmann_Meyer_2019       6
Krasnow_2020               3
Meyer_2019                 3
Misharin_2021              2

sequencing_platform (donors):
sequencing_platform
Illumina NovaSeq 6000                         27
Illumina HiSeq 4000         

/tmp/ipykernel_124/2765457357.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  sites = o.groupby("donor_id")["tissue_level_2"].nunique()
